## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [13]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

groq_api_key=os.getenv("GROQ_API_KEY")


In [14]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001F8191D9460>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F8191D8C50>, model_name='openai/gpt-oss-120b', groq_api_key=SecretStr('**********'))

In [15]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is Krish and I am a Chief AI Engineer")])

AIMessage(content='Hello Krish! It’s great to meet you. As a Chief AI Engineer, you must be working on some fascinating projects. How can I assist you today?', response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 85, 'total_tokens': 179, 'completion_time': 0.202406609, 'completion_tokens_details': {'reasoning_tokens': 52}, 'prompt_time': 0.003424081, 'prompt_tokens_details': None, 'queue_time': 0.045465529, 'total_time': 0.20583069}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e10890e4b9', 'finish_reason': 'stop', 'logprobs': None}, id='run-9ba7525f-ab8f-472a-b2e1-afc0ff9342be-0', usage_metadata={'input_tokens': 85, 'output_tokens': 94, 'total_tokens': 179})

In [16]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Krish and I am a Chief AI Engineer"),
        AIMessage(content="Hello Krish! It's nice to meet you. \n\nAs a Chief AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content='Your name is **Krish**, and you work as a **Chief AI Engineer**.', response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 152, 'total_tokens': 234, 'completion_time': 0.17899744, 'completion_tokens_details': {'reasoning_tokens': 55}, 'prompt_time': 0.01889117, 'prompt_tokens_details': None, 'queue_time': 0.043591324, 'total_time': 0.19788861}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'finish_reason': 'stop', 'logprobs': None}, id='run-a3f3302d-816d-4d83-8ff3-873b1fa09958-0', usage_metadata={'input_tokens': 152, 'output_tokens': 82, 'total_tokens': 234})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [17]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [18]:
config={"configurable":{"session_id":"chat1"}}

In [19]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Krish and I am a Chief AI Engineer")],
    config=config
)

In [20]:
response.content

'Hello Krish! 👋 It’s great to meet you. As a Chief AI Engineer, you’re probably juggling a lot of exciting projects. How can I assist you today? Whether it’s brainstorming ideas, digging into technical details, reviewing code, or anything else, just let me know!'

In [21]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='Your name is Krish.', response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 158, 'total_tokens': 205, 'completion_time': 0.102394156, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.006362392, 'prompt_tokens_details': None, 'queue_time': 0.044126243, 'total_time': 0.108756548}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a09bde29de', 'finish_reason': 'stop', 'logprobs': None}, id='run-261bc48e-b3f3-461f-9471-bb330bc0e19b-0', usage_metadata={'input_tokens': 158, 'output_tokens': 47, 'total_tokens': 205})

In [22]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'I don’t have any record of your name from our conversation so far. If you’d like me to address you by name, just let me know what it is!'

In [23]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
response.content

'Nice to meet you, John! How can I assist you today?'

In [24]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is John.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [25]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [26]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Krish")]})

AIMessage(content='Hello, Krish! Nice to meet you. How can I assist you today?', response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 98, 'total_tokens': 143, 'completion_time': 0.094329987, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.00426457, 'prompt_tokens_details': None, 'queue_time': 0.0492667, 'total_time': 0.098594557}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e10890e4b9', 'finish_reason': 'stop', 'logprobs': None}, id='run-793bdfee-5a45-4e31-b53c-f286d1c64506-0', usage_metadata={'input_tokens': 98, 'output_tokens': 45, 'total_tokens': 143})

In [27]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [30]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Krish")],
    config=config
)

response

AIMessage(content='Hello again, Krish! 👋 Nice to meet you. How can I assist you today?', response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 132, 'total_tokens': 189, 'completion_time': 0.122026781, 'completion_tokens_details': {'reasoning_tokens': 28}, 'prompt_time': 0.005081047, 'prompt_tokens_details': None, 'queue_time': 0.042589482, 'total_time': 0.127107828}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d81b3304b3', 'finish_reason': 'stop', 'logprobs': None}, id='run-e9d8e155-0b07-48af-8da0-f88de1e4b6f6-0', usage_metadata={'input_tokens': 132, 'output_tokens': 57, 'total_tokens': 189})

In [29]:
store

{'chat1': InMemoryChatMessageHistory(messages=[HumanMessage(content='Hi , My name is Krish and I am a Chief AI Engineer'), AIMessage(content='Hello Krish! 👋 It’s great to meet you. As a Chief AI Engineer, you’re probably juggling a lot of exciting projects. How can I assist you today? Whether it’s brainstorming ideas, digging into technical details, reviewing code, or anything else, just let me know!', response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 85, 'total_tokens': 183, 'completion_time': 0.20843875, 'completion_tokens_details': {'reasoning_tokens': 30}, 'prompt_time': 0.014819226, 'prompt_tokens_details': None, 'queue_time': 0.052012884, 'total_time': 0.223257976}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a618bed98', 'finish_reason': 'stop', 'logprobs': None}, id='run-b0e416c9-2334-4b2d-9a3b-8695d06662b8-0', usage_metadata={'input_tokens': 85, 'output_tokens': 98, 'total_tokens': 183}), HumanMessage(content="What's my name?"), AI

In [25]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Krish.  I remember! 😄 \n\nIs there anything else I can help you with?\n'

In [26]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [ ]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Krish")],"language":"Tamil"})
response.content

'नमस्ते कृष्ण! 👋  \n\nमुझे आपसे मिलकर अच्छा लगा।  😊 \n\nआप क्या करना चाहते हैं? \n'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [28]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [29]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Krish")],"language":"Hindi"},
    config=config
)
repsonse.content

'नमस्ते कृष! 😊 \n\nतुम्हें कैसे हो? क्या मैं तुमकी कोई मदद कर सकता हूँ?\n'

In [30]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [31]:
response.content

'आपका नाम कृष है। 😊 \n'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [37]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant"),
 HumanMessage(content='I like vanilla ice cream'),
 AIMessage(content='nice'),
 HumanMessage(content='whats 2 + 2'),
 AIMessage(content='4'),
 HumanMessage(content='thanks'),
 AIMessage(content='no problem!'),
 HumanMessage(content='having fun?'),
 AIMessage(content='yes!')]

In [40]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

"As an AI, I don't have access to your personal preferences like your favorite ice cream flavor.  \n\nWhat's your favorite ice cream? 😊🍦\n"

In [41]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked "whats 2 + 2" 😊  \n\n\n\n'

In [42]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

In [43]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"As a large language model, I don't have access to past conversations or any personal information about you, including your name.  \n\nIf you'd like to tell me your name, I'd be happy to know! 😊  \n\n"

In [44]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

"As a large language model, I have no memory of past conversations. If you'd like to ask me a math problem, I'm happy to help! 😊  Just let me know what it is. \n\n"